<a href="https://colab.research.google.com/github/HectorArielBaez/RegresionAvanzada/blob/main/Voting_(soft)_y_Stacking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()  # selecciona kaggle.json desde tu PC

{}

In [ ]:
# 1. Instalar Kaggle y subir la API token
!pip install kaggle

# Luego, subí tu archivo `kaggle.json` en el directorio raíz de Colab

# 2. Descargar dataset de Kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Por ejemplo, con el dataset 'harunshimanto/epileptic-seizure-recognition'
!kaggle datasets download -d harunshimanto/epileptic-seizure-recognition

# 3. Descomprimir
!unzip -o epileptic-seizure-recognition.zip

ERROR: Operation cancelled by user
Dataset URL: https://www.kaggle.com/datasets/harunshimanto/epileptic-seizure-recognition
License(s): other
epileptic-seizure-recognition.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  epileptic-seizure-recognition.zip
  inflating: Epileptic Seizure Recognition.csv  


In [ ]:
# =========================================================
# 1) Librerías
# =========================================================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

from xgboost import XGBClassifier

In [ ]:
# Cargar dataset
df = pd.read_csv("/content/Epileptic Seizure Recognition.csv")

In [ ]:
# =========================================================
# 3) Preparación del target binario
#    - En este dataset: y=1 (epilepsia), y=2..5 (no epilepsia)
#    - Algunas versiones traen 'y' con 5=not seizure; homogenizamos a 1/0
# =========================================================
if df['y'].nunique() > 2:
    df['y'] = df['y'].apply(lambda v: 1 if v == 1 else 0)  # 1=seizure, otros=0
else:
    # Si ya viene binario, nos aseguramos que la clase positiva sea 1
    df['y'] = (df['y'] == 1).astype(int)

X = df.drop(columns=['y'])
y = df['y']

In [ ]:
# =========================================================
# 4) Split y cálculo de desbalance para XGBoost
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos if pos > 0 else 1.0
print(f"Clase 0 (No epilepsia)={neg} | Clase 1 (Epilepsia)={pos} | scale_pos_weight={scale_pos_weight:.2f}")


Clase 0 (No epilepsia)=6900 | Clase 1 (Epilepsia)=1725 | scale_pos_weight=4.00


In [ ]:
# =========================================================
# 5) Definición de modelos base
#    - LR en pipeline con StandardScaler
#    - RF con class_weight
#    - XGB con scale_pos_weight (para desbalance)
# =========================================================
lr_pipe = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', solver='lbfgs'))
])

rf = RandomForestClassifier(
    n_estimators=300, random_state=42, n_jobs=-1,
    class_weight='balanced', max_depth=None
)

xgb = XGBClassifier(
    n_estimators=400, learning_rate=0.1, max_depth=6,
    subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
    random_state=42, tree_method='hist',
    eval_metric='logloss', scale_pos_weight=scale_pos_weight
)

In [ ]:
# =========================================================
# 6) Voting (soft): promedia probabilidades
# =========================================================
voting_clf = VotingClassifier(
    estimators=[('lr', lr_pipe), ('rf', rf), ('xgb', xgb)],
    voting='soft', n_jobs=-1, weights=None  # podés ajustar weights si querés
)

In [ ]:
# =========================================================
# 7) Stacking: meta-modelo = Regresión Logística
#    - Usa predict_proba (stack_method='predict_proba') para pasar probs al meta-modelo
# =========================================================
stacking_clf = StackingClassifier(
    estimators=[('rf', rf), ('xgb', xgb), ('lr_base', lr_pipe)],
    final_estimator=LogisticRegression(max_iter=2000, class_weight='balanced', solver='lbfgs'),
    stack_method='predict_proba',
    passthrough=False,  # poné True si querés concatenar features originales
    n_jobs=-1, cv=5
)

In [ ]:
# =========================================================
# 8) Entrenamiento
# =========================================================

# Remove the 'Unnamed' column as it contains string values and is not useful for training
X_train = X_train.drop(columns=['Unnamed'])
X_test = X_test.drop(columns=['Unnamed'])

print("\nEntrenando Voting (soft)...")
voting_clf.fit(X_train, y_train)

print("Entrenando Stacking (meta-modelo=LogisticRegression)...")
stacking_clf.fit(X_train, y_train)

print("Entrenando modelos individuales para benchmark...")
lr_pipe.fit(X_train, y_train)
rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)


Entrenando Voting (soft)...
Entrenando Stacking (meta-modelo=LogisticRegression)...
Entrenando modelos individuales para benchmark...


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
# =========================================================
# 9) Evaluación y tabla comparativa
# =========================================================
def eval_model(name, model, X_te, y_te):
    # Algunos modelos son pipelines: predict_proba siempre en el objeto
    y_hat = model.predict(X_te)
    try:
        y_proba = model.predict_proba(X_te)[:, 1]
    except Exception:
        # Para modelos sin predict_proba, usar decision_function si existe
        if hasattr(model, "decision_function"):
            from sklearn.preprocessing import MinMaxScaler
            scores = model.decision_function(X_te).reshape(-1, 1)
            y_proba = MinMaxScaler().fit_transform(scores).ravel()
        else:
            # Fallback binario (no recomendado para AUC)
            y_proba = y_hat.astype(float)

    report = classification_report(y_te, y_hat, output_dict=True, zero_division=0)
    return {
        "Modelo": name,
        "Accuracy": accuracy_score(y_te, y_hat),
        "ROC-AUC": roc_auc_score(y_te, y_proba),
        "Precision (clase 1)": report['1']['precision'],
        "Recall (clase 1)": report['1']['recall'],
        "F1 (clase 1)": report['1']['f1-score']
    }

results = []
results.append(eval_model("Logistic Regression (esc.)", lr_pipe, X_test, y_test))
results.append(eval_model("Random Forest", rf, X_test, y_test))
results.append(eval_model("XGBoost", xgb, X_test, y_test))
results.append(eval_model("Voting (soft)", voting_clf, X_test, y_test))
results.append(eval_model("Stacking (LR meta)", stacking_clf, X_test, y_test))

results_df = pd.DataFrame(results).set_index("Modelo").sort_values("ROC-AUC", ascending=False)
print("\n📊 Comparación de modelos")
display(results_df)


📊 Comparación de modelos


,Accuracy,ROC-AUC,Precision (clase 1),Recall (clase 1),F1 (clase 1)
Modelo,,,,,
Stacking (LR meta),0.975304,0.997116,0.914474,0.966957,0.939983
Random Forest,0.969739,0.996804,0.963878,0.881739,0.920981
XGBoost,0.977043,0.996017,0.953654,0.930435,0.941901
Voting (soft),0.977043,0.995443,0.958559,0.925217,0.941593
Logistic Regression (esc.),0.712696,0.507919,0.327373,0.413913,0.365591


In [ ]:
# =========================================================
# 10) Interpretabilidad:
#     - Coeficientes de la RL (meta-modelo del stacking)
#     - Importancias de RF y XGB
# =========================================================

# a) Coeficientes del meta-modelo (Stacking) → indican peso relativo de las entradas (probs base)
final_lr = stacking_clf.final_estimator_
coef_meta = pd.Series(final_lr.coef_.ravel(), name="Coeficiente (meta LR)")

# Nombramos según los modelos base (solo clase positiva en binario)
coef_meta.index = [f"prob_{name}_cl1" for name, _ in stacking_clf.estimators]

print("\n🔎 Coeficientes del meta-modelo (Logistic Regression en Stacking):")
display(coef_meta)


# b) Importancia de variables (RF)
imp_rf = pd.Series(rf.feature_importances_, index=X_train.columns, name="Importancia_RF") \
           .sort_values(ascending=False).head(15)
print("\n🌳 Top 15 importancias (Random Forest):")
display(imp_rf)

# c) Importancia de variables (XGB)
imp_xgb = pd.Series(xgb.feature_importances_, index=X_train.columns, name="Importancia_XGB") \
            .sort_values(ascending=False).head(15)
print("\n⚡ Top 15 importancias (XGBoost):")
display(imp_xgb)


🔎 Coeficientes del meta-modelo (Logistic Regression en Stacking):


,Coeficiente (meta LR)
prob_rf_cl1,10.605345
prob_xgb_cl1,2.051104
prob_lr_base_cl1,-0.680792



🌳 Top 15 importancias (Random Forest):


,Importancia_RF
X44,0.018402
X156,0.017738
X45,0.017711
X157,0.017168
X160,0.016575
X158,0.014214
X37,0.013407
X43,0.013278
X14,0.012182
X39,0.012032



⚡ Top 15 importancias (XGBoost):


,Importancia_XGB
X44,0.074042
X160,0.069009
X157,0.038826
X159,0.034652
X88,0.029454
X14,0.029428
X12,0.027156
X114,0.018180
X17,0.016698
X92,0.016608
